Visualize the bombcell quality metrics for all sessions! and the thresholds we're applying

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import bombcell as bc
from behave_analysis.postprocess.bc_process.bombcell_utils import get_params, get_metric_info_dict_JR

hist_dict = {}
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"] # list of 6 distinct hex colors
mouse_id = ["JAL003", "JAL004","JAL005","JAL006","JAL007","JAL008"]
metrics = {"nPeaks": 1,
        "nTroughs": 1,
        "waveformBaselineFlatness": .02,
        "waveformDuration_peakTrough": 50,
        "scndPeakToTroughRatio": .02,
        "spatialDecaySlope": .002,
        "peak1ToPeak2Ratio": 5,
        "mainPeakToTroughRatio": .1,
        "rawAmplitude": 15,
        "signalToNoiseRatio": 5,
        "fractionRPVs_estimatedTauR": .05,
        # "nSpikes": 10000,
        "presenceRatio": .01,
        "percentageSpikesMissing_gaussian": 1}



base_path = Path(r"Z:/Jasmine_Laurence/Experimental_Data/")

noks_sessions = []
ks_sessions = []
for mouse in mouse_id:
    mouse_folder = os.path.join(base_path, mouse)
    for session in os.listdir(mouse_folder):
        session_folder = os.path.join(mouse_folder, session)
        # find dir that end in "_g0"
        if os.path.isdir(session_folder):
            for d in os.listdir(session_folder):
                if os.path.isdir(os.path.join(session_folder, d)) and d.endswith("_g0"):
                    imec_folder = os.path.join(session_folder, d, d + "_imec0")
                    if "SI_KS_output" not in os.listdir(imec_folder):
                        noks_sessions.append(os.path.join(session, d, d + "_imec0"))
                    for e in os.listdir(imec_folder):
                        if e == "SI_KS_output":
                            ks_dir = os.path.join(imec_folder, e, "sorter_output")
                            save_path = Path(ks_dir) / "bombcell"
                            if save_path.exists():
                                hist_dict[session] = {}
                                for metric in metrics.keys():
                                    df = pd.read_csv(os.path.join(ks_dir, f"cluster_{metric}.tsv"), sep="\t")
                                    data = df[metric].to_numpy()
                                    h = np.histogram(data, bins=np.arange(np.nanmin(data)-metrics[metric], np.nanmax(data)+metrics[metric], metrics[metric]))
                                    hist_dict[session][metric] = h[0]/np.sum(~np.isnan(data))
                                    hist_dict[session][metric+"_bins"] = h[1][:-1]

"""Load in parmas and quality metrics for one session"""
raw_file = os.path.join(imec_folder, d + "_t0.imec0.ap.bin")
meta_file = os.path.join(imec_folder, d + "_t0.imec0.ap.meta")
param = get_params(ks_dir, raw_file, meta_file, mouse)

"""Make overview plot"""
print("Making overview plot...")
fig = plt.figure(figsize=(20, 10))
gs =gridspec.GridSpec(3,6)
noise_col, mua_col, non_somatic_col = 0, 0, 0
metric_info = get_metric_info_dict_JR(param)
for i, metric in enumerate(metrics.keys()):
    vmi = metric_info[metric]
    if vmi.metric_type == "nonsomatic":
        row = 1
        ax = fig.add_subplot(gs[row, non_somatic_col])
        non_somatic_col += 1
    elif vmi.metric_type == "noise":
        row = 0
        ax = fig.add_subplot(gs[row, noise_col])
        noise_col += 1
    elif vmi.metric_type == "mua":
        row = 2
        ax = fig.add_subplot(gs[row, mua_col])
        mua_col += 1
    for session in hist_dict.keys():
        # check which mouse the sessions belongs to by looking for 003, 004, 005, 006, 007, 008 and use a different color for each mouse
        m = [m for m,i in enumerate(mouse_id) if i[-3:] in session][0]
        if len(hist_dict[session][metric]) <3:
            ax.bar(hist_dict[session][metric+"_bins"], hist_dict[session][metric], width=metrics[metric], ec=colors[m], facecolor = 'none')
        else:
            ax.plot(hist_dict[session][metric+"_bins"], hist_dict[session][metric], color=colors[m])
    ax.set_xlabel(vmi.short_name)
    patchx = np.array(ax.get_xlim())
    if vmi.min_threshold is not None:
        patchx[0] = vmi.min_threshold
    if vmi.max_threshold is not None:
        patchx[1] = vmi.max_threshold
    patch = plt.Rectangle((patchx[0], 0), patchx[1]-patchx[0], 1.1, color="green", alpha=0.3)
    ax.add_patch(patch)   
    ax.set_ylim(0,1.1)
    if metric == "rawAmplitude":
        ax.set_xlim(0, 200)
    if metric == "signalToNoiseRatio":
        ax.set_xlim(0, 100)

fig.savefig("Z:/Jasmine_Laurence/HPC/bombcell/bombcell_overview_plot.png")